In [ ]:
# =============================================================================
# maskrcnn_detection_segmentation.py
# Mask R-CNN: Multi-class clothing detection + instance segmentation
# on DeepFashion2 (top-5 categories)
#
# Dataset structure (same as classification notebook):
#   /kaggle/input/.../final_dataset/train/images/       <- .jpg
#   /kaggle/input/.../final_dataset/train/annotations/  <- .json
#   /kaggle/input/.../final_dataset/val/images/         <- .jpg
#   /kaggle/input/.../final_dataset/val/annotations/    <- .json
#
# Two training strategies:
#   Transfer Learning : COCO-pretrained backbone + replaced heads
#   From Scratch      : randomly initialized weights (baseline)
#
# Evaluation:
#   Detection  : mAP@[0.5:0.95], per-class ROC/AUC, F1-score
#   Segmentation: mIoU (per-class + macro), Dice coefficient
#
# NOTE: 50% subsampling applied — every other image is used from
#       both train and val splits to reduce compute requirements.
#       The 80/20 split ratio is preserved before subsampling.
# =============================================================================


# =============================================================================
# SECTION 1: INSTALL DEPENDENCIES & IMPORTS
# =============================================================================

import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except ImportError:
    pip_install("torchmetrics[detection]")
    from torchmetrics.detection.mean_ap import MeanAveragePrecision

import os
import json
import pickle
import random
import time
import copy
import numpy as np
import matplotlib
matplotlib.use("Agg")               # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn,
    MaskRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_curve, auc, average_precision_score,
)

print(f"PyTorch    : {torch.__version__}")
print(f"Torchvision: {torchvision.__version__}")
print(f"CUDA       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device     : {DEVICE}")


# =============================================================================
# SECTION 2: PATHS & HYPER-PARAMETERS
# =============================================================================

# Using the exact paths copied from Kaggle UI
SRC_BASE  = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/pruned_deepfashion2/final_dataset"
SPLIT_PKL = "/kaggle/input/datasets/kavyagupta3011/vr-dataset-deepfashion2/split.pkl"

SRC_TRAIN_IMG = os.path.join(SRC_BASE, "train", "images")
SRC_TRAIN_ANN = os.path.join(SRC_BASE, "train", "annotations")
SRC_VAL_IMG   = os.path.join(SRC_BASE, "val",   "images")
SRC_VAL_ANN   = os.path.join(SRC_BASE, "val",   "annotations")

IMG_SIZE             = 512    
BATCH_SIZE           = 2      
NUM_EPOCHS_TRANSFER  = 8
NUM_EPOCHS_SCRATCH   = 8
LR_TRANSFER          = 5e-4
LR_SCRATCH           = 1e-3
WEIGHT_DECAY         = 1e-4
GRAD_CLIP            = 1.0    
MASK_THRESHOLD       = 0.5    
IOU_THRESHOLD        = 0.5    
SCORE_THRESHOLD      = 0.3    

SUBSAMPLE_STEP = 3          

OUTPUT_DIR  = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =============================================================================
# SECTION 3: LABEL MAP & CATEGORY SETUP
# =============================================================================

# Fixed 0-based label map (same as classification notebook)
LABEL_MAP = {
    "short sleeve top": 0,
    "trousers":         1,
    "shorts":           2,
    "long sleeve top":  3,
    "skirt":            4,
}

# Mask R-CNN reserves class 0 for background → clothing labels are 1..5
top5         = {1, 8, 7, 2, 9}               # DeepFashion2 category_ids
cat_to_index = {1: 0, 8: 1, 7: 2, 2: 3, 9: 4}    # cat_id  → 0-based
index_to_cat = {0: "short sleeve top", 1: "trousers",
                2: "shorts",           3: "long sleeve top", 4: "skirt"}
cat_to_label = {cid: idx + 1 for cid, idx in cat_to_index.items()}  # cat_id → 1-based
label_to_idx = {lbl: lbl - 1 for lbl in cat_to_label.values()}       # 1-based → 0-based
num_classes  = 5              # number of clothing classes (excl. background)
num_labels   = num_classes + 1    # total Mask R-CNN classes (incl. background)

print("Category label mapping (0 = background in MaskRCNN):")
for cid, lbl in cat_to_label.items():
    print(f"  category_id={cid}  label={lbl}  ({index_to_cat[cat_to_index[cid]]})")

with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w") as f:
    json.dump(LABEL_MAP, f, indent=4)
print("Saved label_map.json")



# =============================================================================
# SECTION 4: 80 / 20 DATASET SPLIT  +  50% SUBSAMPLING
#
# CHANGE vs original:
#   After loading/computing the 80/20 split we apply a stride-2 slice
#   (train_files[::2] and val_files[::2]) so that only 1 in every 2
#   images is used.  This halves both splits while preserving their
#   relative 80/20 ratio and the original class distribution ordering.
#
# split.pkl is reused from the classification notebook when present
# (it stores the full lists; subsampling is applied in-memory here).
# =============================================================================

def split_train_folder(val_size=0.20, seed=42):
    """
    Compute stratified 80/20 index lists from the source train folder.
    Returns FULL lists — subsampling is applied by the caller.
    """
    valid_files, strat_labels = [], []
    ann_files = sorted(os.listdir(SRC_TRAIN_ANN))
    print(f"Scanning {len(ann_files)} annotation files...")

    for ann_file in ann_files:
        if not ann_file.endswith(".json"):
            continue
        img_file = ann_file.replace(".json", ".jpg")
        if not os.path.exists(os.path.join(SRC_TRAIN_IMG, img_file)):
            continue

        with open(os.path.join(SRC_TRAIN_ANN, ann_file)) as f:
            data = json.load(f)

        cats = [v["category_id"] for v in data.values()
                if isinstance(v, dict) and v.get("category_id") in top5]
        if not cats:
            continue

        valid_files.append(img_file)
        strat_labels.append(max(set(cats), key=cats.count))

    print(f"Valid images (full): {len(valid_files)}")
    tr, vl = train_test_split(
        valid_files, test_size=val_size, stratify=strat_labels, random_state=seed
    )
    print(f"  Train (80%): {len(tr)}  |  Val (20%): {len(vl)}")
    return tr, vl


# ── Load or compute full split ────────────────────────────────────────────
if os.path.exists(SPLIT_PKL):
    with open(SPLIT_PKL, "rb") as f:
        saved = pickle.load(f)
    train_files_full = saved["train_files"]
    val_files_full   = saved["val_files"]
    print(f"Loaded split.pkl — {len(train_files_full)} train, "
          f"{len(val_files_full)} val (FULL lists)")
else:
    print("split.pkl not found — recomputing split...")
    train_files_full, val_files_full = split_train_folder()
    with open(SPLIT_PKL, "wb") as f:
        pickle.dump({"train_files": train_files_full,
                     "val_files":   val_files_full}, f)
    print("Saved new split.pkl")

# ── 50% SUBSAMPLING: take every other image ───────────────────────────────
# Slicing with [::SUBSAMPLE_STEP] picks index 0, 2, 4, … from each list.
# Because train_test_split already shuffled/stratified the lists, this
# sub-selection is representative of the full class distribution.
train_files = train_files_full[::SUBSAMPLE_STEP]   # ~40% of original dataset
val_files   = val_files_full[::SUBSAMPLE_STEP]     # ~10% of original dataset

print(f"\n50% subsampling applied (step={SUBSAMPLE_STEP}):")
print(f"  train_files : {len(train_files_full):6d}  →  {len(train_files):6d} images")
print(f"  val_files   : {len(val_files_full):6d}  →  {len(val_files):6d} images")

# Provided val folder (final evaluation only — never used during training)
# We also subsample the eval set consistently for faster evaluation
val_eval_files_full = sorted([f for f in os.listdir(SRC_VAL_IMG)
                               if f.endswith(".jpg")])
val_eval_files = val_eval_files_full[::SUBSAMPLE_STEP]
print(f"  val_eval    : {len(val_eval_files_full):6d}  →  {len(val_eval_files):6d} images "
      f"(provided val folder)")


# =============================================================================
# SECTION 5: CLASS INSTANCE COUNT & OVERSAMPLING WEIGHTS
# Computed from the already-subsampled train_files list.
# =============================================================================

def compute_sample_weights(file_list, ann_folder):
    """
    Compute a per-sample weight for WeightedRandomSampler.
    A sample's weight = max(weight over all its categories).
    """
    class_counts = np.zeros(num_classes, dtype=float)

    for img_file in file_list:
        ann_path = os.path.join(ann_folder, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                class_counts[cat_to_index[v["category_id"]]] += 1

    class_weights = 1.0 / (class_counts + 1e-6)      # inverse freq
    class_weights /= class_weights.sum()              # normalise

    sample_weights = []
    for img_file in file_list:
        ann_path = os.path.join(ann_folder, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        w = 0.0
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                w = max(w, class_weights[cat_to_index[v["category_id"]]])
        sample_weights.append(w if w > 0 else 1e-6)

    print("\nClass instance counts (subsampled train split):")
    for i in range(num_classes):
        print(f"  [{i}] {index_to_cat[i]:25s}  count={int(class_counts[i]):6d}"
              f"  weight={class_weights[i]:.6f}")
    return np.array(sample_weights, dtype=float)


sample_weights = compute_sample_weights(train_files, SRC_TRAIN_ANN)


# =============================================================================
# SECTION 6: HELPER — POLYGON → BINARY MASK
# =============================================================================

def polygons_to_mask(polygons, height, width):
    """
    Convert a list of flat-coordinate polygons to a single (H, W) uint8 mask.
    polygons: [[x1,y1,x2,y2,...], [...], ...]  (pixel coords in original space)
    height/width: size of the output mask (should match resized image)
    """
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    for poly in polygons:
        if len(poly) < 6:          # need at least 3 points
            continue
        coords = list(zip(poly[0::2], poly[1::2]))
        draw.polygon(coords, fill=1)
    return np.array(mask, dtype=np.uint8)


# =============================================================================
# SECTION 7: PYTORCH DATASET
# =============================================================================

class DeepFashion2Dataset(Dataset):
    """
    PyTorch Dataset for DeepFashion2 instance segmentation + detection.

    Each __getitem__ returns:
        image  : FloatTensor [3, IMG_SIZE, IMG_SIZE]  values in [0, 1]
        target : dict with keys
                   boxes    - FloatTensor [N, 4]  (x1,y1,x2,y2)
                   labels   - LongTensor  [N]     1-indexed (0 = background)
                   masks    - ByteTensor  [N, H, W]
                   area     - FloatTensor [N]
                   image_id - LongTensor  [1]
                   iscrowd  - LongTensor  [N]  (all 0)
    """

    def __init__(self, file_list, img_folder, ann_folder,
                 augment=False, img_size=IMG_SIZE):
        self.file_list  = list(file_list)
        self.img_folder = img_folder
        self.ann_folder = ann_folder
        self.augment    = augment
        self.img_size   = img_size

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_file = self.file_list[idx]
        img_path = os.path.join(self.img_folder, img_file)
        ann_path = os.path.join(self.ann_folder, img_file.replace(".jpg", ".json"))

        # ── load & resize image ────────────────────────────────────────────
        img = Image.open(img_path).convert("RGB")
        orig_w, orig_h = img.size
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_tensor = TF.to_tensor(img)           # [3, H, W], float [0,1]

        scale_x = self.img_size / orig_w
        scale_y = self.img_size / orig_h

        # ── parse annotation ───────────────────────────────────────────────
        with open(ann_path) as f:
            data = json.load(f)

        boxes, labels, masks = [], [], []

        for v in data.values():
            if not isinstance(v, dict):
                continue
            cat_id = v.get("category_id")
            if cat_id not in top5:
                continue

            # ── bounding box ───────────────────────────────────────────────
            x1, y1, x2, y2 = v["bounding_box"]
            x1 = float(x1) * scale_x
            y1 = float(y1) * scale_y
            x2 = float(x2) * scale_x
            y2 = float(y2) * scale_y

            # clamp to image boundary
            x1 = max(0.0, min(x1, self.img_size - 1))
            y1 = max(0.0, min(y1, self.img_size - 1))
            x2 = max(0.0, min(x2, self.img_size))
            y2 = max(0.0, min(y2, self.img_size))

            if (x2 - x1) < 1.0 or (y2 - y1) < 1.0:
                continue          # degenerate box — skip

            # ── segmentation mask ──────────────────────────────────────────
            segs = v.get("segmentation", [])
            # scale polygon coords to resized image space
            scaled_segs = []
            for poly in segs:
                scaled = []
                for i, coord in enumerate(poly):
                    scaled.append(coord * scale_x if i % 2 == 0
                                  else coord * scale_y)
                scaled_segs.append(scaled)

            mask = polygons_to_mask(scaled_segs, self.img_size, self.img_size)

            boxes.append([x1, y1, x2, y2])
            labels.append(cat_to_label[cat_id])   # 1-indexed
            masks.append(mask)

        # ── build target tensors ───────────────────────────────────────────
        if len(boxes) == 0:
            boxes_t  = torch.zeros((0, 4),  dtype=torch.float32)
            labels_t = torch.zeros((0,),    dtype=torch.int64)
            masks_t  = torch.zeros((0, self.img_size, self.img_size),
                                   dtype=torch.uint8)
            area_t   = torch.zeros((0,),    dtype=torch.float32)
        else:
            boxes_t  = torch.tensor(boxes,           dtype=torch.float32)
            labels_t = torch.tensor(labels,          dtype=torch.int64)
            masks_t  = torch.tensor(np.stack(masks), dtype=torch.uint8)
            area_t   = ((boxes_t[:, 2] - boxes_t[:, 0]) *
                        (boxes_t[:, 3] - boxes_t[:, 1]))

        target = {
            "boxes":    boxes_t,
            "labels":   labels_t,
            "masks":    masks_t,
            "area":     area_t,
            "image_id": torch.tensor([idx], dtype=torch.int64),
            "iscrowd":  torch.zeros(len(boxes_t), dtype=torch.int64),
        }

        # ── augmentation (training only) ───────────────────────────────────
        if self.augment:
            img_tensor, target = self._augment(img_tensor, target)

        return img_tensor, target

    # ── augmentation helper ────────────────────────────────────────────────
    @staticmethod
    def _augment(image, target):
        """Geometric + colour augmentations that keep boxes/masks consistent."""
        w = image.shape[-1]

        # Random horizontal flip
        if random.random() > 0.5:
            image = TF.hflip(image)
            if target["boxes"].numel() > 0:
                boxes = target["boxes"].clone()
                boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                target["boxes"] = boxes
            if target["masks"].numel() > 0:
                target["masks"] = target["masks"].flip(-1)

        # Colour jitter (image only — no effect on boxes/masks)
        image = TF.adjust_brightness(image, 1.0 + random.uniform(-0.15, 0.15))
        image = TF.adjust_contrast(image,   1.0 + random.uniform(-0.15, 0.15))
        image = TF.adjust_saturation(image, 1.0 + random.uniform(-0.10, 0.10))

        return image, target


def collate_fn(batch):
    """Custom collate: keep images and targets as tuples (variable N per image)."""
    return tuple(zip(*batch))

# =============================================================================
# SECTION 8: DATALOADERS
# =============================================================================

print("\nBuilding DataLoaders...")

# WeightedRandomSampler for train set (handles class imbalance)
train_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_files),
    replacement=True
)

train_dataset    = DeepFashion2Dataset(
    train_files, SRC_TRAIN_IMG, SRC_TRAIN_ANN, augment=True)
val_fit_dataset  = DeepFashion2Dataset(
    val_files, SRC_TRAIN_IMG, SRC_TRAIN_ANN, augment=False)
val_eval_dataset = DeepFashion2Dataset(
    val_eval_files, SRC_VAL_IMG, SRC_VAL_ANN, augment=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=train_sampler, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)
val_fit_loader = DataLoader(
    val_fit_dataset, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)
val_eval_loader = DataLoader(
    val_eval_dataset, batch_size=BATCH_SIZE,
    shuffle=False, collate_fn=collate_fn,
    num_workers=2, pin_memory=True
)

print(f"  train_loader    : {len(train_loader)} batches  "
      f"({len(train_dataset)} images  ← 50% of train 80%)")
print(f"  val_fit_loader  : {len(val_fit_loader)} batches  "
      f"({len(val_fit_dataset)} images  ← 50% of val 20%)")
print(f"  val_eval_loader : {len(val_eval_loader)} batches  "
      f"({len(val_eval_dataset)} images  ← 50% of provided val folder)")



# =============================================================================
# SECTION 9: DATASET DISTRIBUTION VISUALIZATION
# =============================================================================

def plot_dataset_distribution():
    def count_instances(file_list, ann_folder):
        counts = np.zeros(num_classes, dtype=int)
        for f in file_list:
            ann = os.path.join(ann_folder, f.replace(".jpg", ".json"))
            with open(ann) as fp:
                data = json.load(fp)
            for v in data.values():
                if isinstance(v, dict) and v.get("category_id") in top5:
                    counts[cat_to_index[v["category_id"]]] += 1
        return counts

    print("Counting instance distribution (subsampled splits)...")
    tr_cnt = count_instances(train_files,    SRC_TRAIN_ANN)
    vf_cnt = count_instances(val_files,      SRC_TRAIN_ANN)
    ve_cnt = count_instances(val_eval_files, SRC_VAL_ANN)

    x      = np.arange(num_classes)
    w      = 0.28
    labels = [index_to_cat[i] for i in range(num_classes)]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w, tr_cnt, w, label="Train 50% subsample", color="#4e79a7")
    ax.bar(x,     vf_cnt, w, label="Val 50% subsample",   color="#f28e2b")
    ax.bar(x + w, ve_cnt, w, label="Eval Val 50%",        color="#59a14f")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("Instance count")
    ax.set_title("Instance Distribution — 50% Subsampled Splits (top-5 categories)")
    ax.legend()
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "maskrcnn_split_distribution.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")

plot_dataset_distribution()



# =============================================================================
# SECTION 10: AUGMENTATION VISUALIZATION
# Shows one example image per category with augmentations applied.
# =============================================================================

def visualize_augmentations():
    """Show original and 3 augmented versions for one image per category."""
    samples = {}
    for img_file in train_files:
        ann_path = os.path.join(SRC_TRAIN_ANN, img_file.replace(".jpg", ".json"))
        with open(ann_path) as f:
            data = json.load(f)
        for v in data.values():
            if isinstance(v, dict) and v.get("category_id") in top5:
                idx = cat_to_index[v["category_id"]]
                if idx not in samples:
                    samples[idx] = img_file
        if len(samples) == num_classes:
            break

    fig, axes = plt.subplots(num_classes, 4,
                              figsize=(16, num_classes * 3.5))

    for row, (cat_idx, img_file) in enumerate(sorted(samples.items())):
        img   = Image.open(os.path.join(SRC_TRAIN_IMG, img_file)).convert("RGB")
        img   = img.resize((IMG_SIZE, IMG_SIZE))
        img_t = TF.to_tensor(img)

        # Original
        axes[row][0].imshow(img)
        axes[row][0].set_title(f"{index_to_cat[cat_idx]}\nOriginal", fontsize=8)
        axes[row][0].axis("off")

        # 3 augmented versions
        for col in range(1, 4):
            dummy_target = {
                "boxes":    torch.zeros((0, 4)),
                "labels":   torch.zeros((0,), dtype=torch.int64),
                "masks":    torch.zeros((0, IMG_SIZE, IMG_SIZE), dtype=torch.uint8),
                "area":     torch.zeros((0,)),
                "image_id": torch.tensor([0]),
                "iscrowd":  torch.zeros((0,), dtype=torch.int64),
            }
            aug_t, _ = DeepFashion2Dataset._augment(img_t.clone(), dummy_target)
            aug_np   = aug_t.permute(1, 2, 0).numpy()
            aug_np   = (aug_np * 255).clip(0, 255).astype(np.uint8)
            axes[row][col].imshow(aug_np)
            axes[row][col].set_title(f"Augmented #{col}", fontsize=8)
            axes[row][col].axis("off")

    plt.suptitle("Data Augmentation Preview (per category)", fontsize=13)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "maskrcnn_augmentation_preview.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved {path}")

visualize_augmentations()


# =============================================================================
# SECTION 11: SANITY CHECK — VISUALIZE ONE BATCH WITH BOXES & MASKS
# =============================================================================

def visualize_batch(loader, title="Batch Sample", save_name="batch_sample.png",
                    n_images=4):
    images, targets = next(iter(loader))
    colors     = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00"]
    cat_colors = {i + 1: colors[i] for i in range(num_classes)}

    fig, axes = plt.subplots(1, min(n_images, len(images)),
                              figsize=(5 * min(n_images, len(images)), 5))
    if not isinstance(axes, np.ndarray):
        axes = [axes]

    for ax, img_t, tgt in zip(axes, images[:n_images], targets[:n_images]):
        img_np = img_t.permute(1, 2, 0).numpy()
        img_np = (img_np * 255).clip(0, 255).astype(np.uint8)
        ax.imshow(img_np)

        for box, lbl, mask in zip(tgt["boxes"], tgt["labels"], tgt["masks"]):
            x1, y1, x2, y2 = box.tolist()
            lbl   = lbl.item()
            color = cat_colors.get(lbl, "white")
            rect  = plt.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, index_to_cat.get(lbl - 1, str(lbl)),
                    color=color, fontsize=7, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.1", fc="black", alpha=0.5))

            # overlay mask
            mask_np = mask.numpy().astype(bool)
            overlay = np.zeros((*mask_np.shape, 4), dtype=float)
            rgb = tuple(int(color.lstrip("#")[i:i+2], 16) / 255.0
                        for i in (0, 2, 4))
            overlay[mask_np] = [*rgb, 0.35]
            ax.imshow(overlay)

        ax.axis("off")

    legend = [mpatches.Patch(color=cat_colors[i+1], label=index_to_cat[i])
              for i in range(num_classes)]
    fig.legend(handles=legend, loc="lower center",
               ncol=num_classes, fontsize=9)
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {path}")

print("\nVisualizing a training batch...")
visualize_batch(train_loader,
                title="Training Batch — GT Boxes & Masks (50% subsample)",
                save_name="maskrcnn_train_batch_sample.png")




# =============================================================================
# SECTION 12: MODEL BUILDING
# =============================================================================

def build_maskrcnn(strategy="transfer"):
    """
    Build Mask R-CNN with ResNet-50 FPN backbone.

    strategy="transfer":
        - Backbone initialized from COCO pretrained weights
        - Both box predictor and mask predictor replaced for num_labels classes
        - Backbone initially frozen; unfreeze_backbone() is called after warm-up
    strategy="scratch":
        - All weights randomly initialized
        - All layers trainable from the start
    """
    if strategy == "transfer":
        model = maskrcnn_resnet50_fpn(
            weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        )
        print("Transfer: loaded COCO-pretrained Mask R-CNN weights")
    elif strategy == "scratch":
        model = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None)
        print("Scratch: randomly initialized Mask R-CNN (no pretrained weights)")
    else:
        raise ValueError("strategy must be 'transfer' or 'scratch'")

    # ── Replace box predictor ──────────────────────────────────────────────
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_labels)

    # ── Replace mask predictor ─────────────────────────────────────────────
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, 256, num_labels
    )

    # ── Freeze backbone for transfer learning (warm-up phase) ─────────────
    if strategy == "transfer":
        for param in model.backbone.parameters():
            param.requires_grad = False
        print("Backbone frozen for warm-up (will unfreeze after warm-up epochs)")

    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total params    : {total:,}")
    print(f"  Trainable params: {trainable:,}")
    return model.to(DEVICE)


def unfreeze_backbone(model):
    """Unfreeze backbone layers for fine-tuning stage."""
    for param in model.backbone.parameters():
        param.requires_grad = True
    print("Backbone unfrozen for fine-tuning.")


# =============================================================================
# SECTION 13: TRAINING UTILITIES
# =============================================================================

def train_one_epoch(model, optimizer, loader, epoch, total_epochs,
                    scheduler=None):
    """Run one training epoch; returns dict of mean losses."""
    model.train()
    loss_accum = defaultdict(float)
    n_batches  = 0
    t0         = time.time()

    for batch_idx, (images, targets) in enumerate(loader):
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        # Forward pass (training mode returns loss dict)
        loss_dict = model(images, targets)
        losses    = sum(loss_dict.values())

        # Backward
        optimizer.zero_grad()
        losses.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()

        for k, v in loss_dict.items():
            loss_accum[k] += v.item()
        loss_accum["total"] += losses.item()
        n_batches += 1

        if (batch_idx + 1) % max(1, len(loader) // 5) == 0:
            elapsed = time.time() - t0
            print(f"  Epoch [{epoch}/{total_epochs}] "
                  f"Batch [{batch_idx+1}/{len(loader)}] "
                  f"Loss: {loss_accum['total']/n_batches:.4f}  "
                  f"({elapsed:.1f}s)")

    return {k: v / n_batches for k, v in loss_accum.items()}


@torch.no_grad()
def validate_loss(model, loader):
    """Compute mean losses on a validation set (model in train mode for loss)."""
    model.train()           # torchvision Mask R-CNN needs train mode for loss dict
    loss_accum = defaultdict(float)
    n_batches  = 0

    for images, targets in loader:
        images  = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        for k, v in loss_dict.items():
            loss_accum[k] += v.item()
        loss_accum["total"] += sum(loss_dict.values()).item()
        n_batches += 1

    return {k: v / n_batches for k, v in loss_accum.items()}



# =============================================================================
# SECTION 13.5: SMOKE TEST (Sanity Check before full training)
# =============================================================================
print("\n" + "="*60)
print("RUNNING SMOKE TEST (1 Batch Forward & Backward Pass)")
print("="*60)

try:
    # 1. Build a temporary model
    smoke_model = build_maskrcnn(strategy="scratch")
    smoke_model.train()
    
    # 2. Setup a basic optimizer
    smoke_optim = torch.optim.AdamW(smoke_model.parameters(), lr=1e-4)
    
    # 3. Get exactly ONE batch from the train loader
    print("  Fetching a training batch...")
    images, targets = next(iter(train_loader))
    images  = [img.to(DEVICE) for img in images]
    targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
    
    # 4. Forward Pass
    print("  Testing Forward Pass (Loss calculation)...")
    loss_dict = smoke_model(images, targets)
    losses    = sum(loss_dict.values())
    
    # 5. Backward Pass
    print("  Testing Backward Pass (Gradients)...")
    smoke_optim.zero_grad()
    losses.backward()
    smoke_optim.step()
    
    # 6. Test Validation Check (Ensure no eval mode crashes)
    print("  Testing Validation Loop (1 batch)...")
    # We just borrow the next batch from the val_fit_loader manually for speed
    val_images, val_targets = next(iter(val_fit_loader))
    val_images  = [img.to(DEVICE) for img in val_images]
    val_targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in val_targets]
    _ = smoke_model(val_images, val_targets)
    
    print("  ✓ SMOKE TEST PASSED! The pipeline is completely healthy.")
    
    # 7. CRITICAL: Clean up memory so the real training has full GPU capacity
    del smoke_model, smoke_optim, images, targets, loss_dict, losses
    del val_images, val_targets
    torch.cuda.empty_cache()

except Exception as e:
    print(f"\n  SMOKE TEST FAILED! Fix the error below before training:\n")
    raise e  # Halts the script so you don't waste time/compute




# =============================================================================
# SECTION 15: TRAINING — FROM SCRATCH (BASELINE)
# =============================================================================

print("\n" + "="*60)
print("TRAINING: From Scratch (random init baseline)")
print(f"  Dataset : {len(train_files)} train images (50% subsample)")
print("="*60)

model_scratch = build_maskrcnn(strategy="scratch")

optimizer_sc = torch.optim.AdamW(
    model_scratch.parameters(), lr=LR_SCRATCH, weight_decay=WEIGHT_DECAY
)
scheduler_sc = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_sc, T_max=NUM_EPOCHS_SCRATCH
)

history_scratch  = {"train_loss": [], "val_loss": []}
best_val_loss_sc = float("inf")
best_state_sc    = None

for epoch in range(1, NUM_EPOCHS_SCRATCH + 1):
    print(f"\n--- Scratch Epoch {epoch}/{NUM_EPOCHS_SCRATCH} ---")
    train_losses = train_one_epoch(
        model_scratch, optimizer_sc, train_loader,
        epoch, NUM_EPOCHS_SCRATCH, scheduler_sc
    )
    val_losses = validate_loss(model_scratch, val_fit_loader)

    history_scratch["train_loss"].append(train_losses["total"])
    history_scratch["val_loss"].append(val_losses["total"])

    print(f"  Train Loss: {train_losses['total']:.4f}  "
          f"(cls={train_losses.get('loss_classifier', 0):.3f}  "
          f"box={train_losses.get('loss_box_reg', 0):.3f}  "
          f"mask={train_losses.get('loss_mask', 0):.3f})")
    print(f"  Val   Loss: {val_losses['total']:.4f}")

    if val_losses["total"] < best_val_loss_sc:
        best_val_loss_sc = val_losses["total"]
        best_state_sc    = copy.deepcopy(model_scratch.state_dict())
        torch.save(best_state_sc,
                   os.path.join(OUTPUT_DIR, "best_maskrcnn_scratch.pth"))
        print(f"  ✓ New best val loss: {best_val_loss_sc:.4f} — checkpoint saved")

print("\nFrom-scratch training complete.")
model_scratch.load_state_dict(best_state_sc)



# =============================================================================
# SECTION 16: SAVE SCRATCH MODEL & HISTORY
# =============================================================================
print("\nSaving From-Scratch model and training history...")

# Save the weights
torch.save(model_scratch.state_dict(),
           os.path.join(OUTPUT_DIR, "maskrcnn_scratch_final.pth"))

# Save the config
config = {
    "model_type":      "maskrcnn_resnet50_fpn",
    "num_classes":     num_classes,
    "num_labels":      num_labels,
    "img_size":        IMG_SIZE,
    "mask_threshold":  MASK_THRESHOLD,
    "score_threshold": SCORE_THRESHOLD,
    "label_map":       LABEL_MAP,
    "cat_to_label":    {str(k): v for k, v in cat_to_label.items()},
    "index_to_cat":    {str(k): v for k, v in index_to_cat.items()},
    "subsample_step":  SUBSAMPLE_STEP,
}
with open(os.path.join(OUTPUT_DIR, "maskrcnn_config.json"), "w") as f:
    json.dump(config, f, indent=4)

# Save the training history for plotting later
with open(os.path.join(OUTPUT_DIR, "history_scratch.json"), "w") as f:
    json.dump(history_scratch, f, indent=4)

print("\nAll outputs saved to", OUTPUT_DIR)
print("Notebook 1 Complete!")